# Asset Failure - Survival Model - Procurment Algorithm

#### Author - Abhinav Paul

In [3]:
# Import necessary libraries for data manipulation, visualization, and survival analysis
! pip install lifelines scikit-learn matplotlib seaborn scikit-survival

# Core data science libraries
import pandas as pd
# Configure pandas display options for better data exploration
pd.set_option("display.max_rows", 200)  # Show more rows in output
pd.set_option("display.max_columns", 50)  # Show more columns
pd.set_option("display.width", 1000)  # Wider display
pd.set_option("display.max_colwidth", None)  # Show full column content

import numpy as np  # Numerical operations
import os  # Operating system utilities
import pickle  # For serializing/deserializing Python objects

# Visualization libraries
import matplotlib.pyplot as plt  # Basic plotting
import seaborn as sns  # Statistical visualization

# Survival Analysis libraries
from lifelines.statistics import logrank_test  # Statistical tests for survival curves
from lifelines import KaplanMeierFitter, CoxPHFitter  # Non-parametric and semi-parametric models
from sksurv.ensemble import RandomSurvivalForest  # Machine learning approach to survival
from sksurv.util import Surv  # Utility for creating survival data structures
from sklearn.inspection import permutation_importance  # Feature importance evaluation
from sksurv.metrics import concordance_index_censored  # Model evaluation metric

# Data preprocessing
from sklearn.preprocessing import StandardScaler  # Feature scaling

# Date handling
from datetime import datetime, timedelta  # Date manipulation utilities

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

### Load Data

Load all necessary data sources for the survival analysis:
- inventory: Asset specifications and stock information
- assets: Asset metadata including warranty and installation dates


In [4]:
# Load data from pickle files (efficient binary format for pandas DataFrames)
# Using pickle format preserves data types and is faster than CSV for large datasets
inventory = pd.read_pickle('inventory.pkl')
assets_to_order_now = pd.read_pickle('assets_to_order_now.pkl')

#### Inventory Aggregation

Inventory data provides static asset characteristics that can influence survival:
1. Device type, manufacturer, model: Different hardware may have different reliability
2. Unit cost: More expensive assets may have better maintenance
3. Warehouse location: Environmental factors may affect asset lifespan
4. Lead time: Replacement time affects business impact of failures

Statistical value: These static covariates help explain baseline hazard differences
between asset types and improve prediction accuracy.

In [5]:
# Examine the inventory data structure
# Shows static asset characteristics that will be used as baseline covariates
# in the survival analysis model
inventory.head()

,asset_id,device_type,manufacturer,model_number,warehouse_location,current_stock_quantity,reorder_threshold_quantity,unit_cost,safety_stock_quantity,lead_time_days,file_year
0,AID_001,Server,Dell,M001,WH1,56,40,2753.0,17,6,2023
1,AID_002,Server,Dell,M002,WH1,76,40,4967.0,12,12,2023
2,AID_003,Server,Dell,M003,WH2,58,40,744.0,12,6,2023
3,AID_004,Server,Dell,M004,WH1,179,40,717.0,27,6,2023
4,AID_005,Server,Cisco,M001,WH2,106,40,4179.0,28,7,2023


In [6]:
# Get the latest inventory record for each asset
# Statistical reasoning: Use the most recent inventory data as it represents the current
# asset specifications and stock levels, which are most relevant for current survival predictions

latest_inventory = inventory.sort_values(['asset_id', 'file_year']).groupby('asset_id').tail(1).reset_index(drop=True)
del inventory

# Data quality check: Ensure we have exactly one record per asset
# Duplicate asset records would indicate data quality issues
assert latest_inventory['asset_id'].count() - latest_inventory['asset_id'].nunique() == 0, f"""
Duplicate asset_id detected!
Total rows: {latest_inventory['asset_id'].count()}
Unique IDs: {latest_inventory['asset_id'].nunique()}
Duplicates: {latest_inventory['asset_id'].count() - latest_inventory['asset_id'].nunique()}
"""

# Verify the latest inventory data for our test asset (AID_053)
latest_inventory[latest_inventory['asset_id'] == 'AID_053'].head(20).sort_values(['asset_id'])

,asset_id,device_type,manufacturer,model_number,warehouse_location,current_stock_quantity,reorder_threshold_quantity,unit_cost,safety_stock_quantity,lead_time_days,file_year
52,AID_053,Switch,Cisco,M001,WH2,119,40,4535.99,19,10,2025


In [7]:
# Verify that we have the latest inventory data for all assets
# All assets should have 2025 data, indicating we're using the most recent information
latest_inventory.file_year.value_counts()

,count
file_year,
2025,64


In [8]:
# Group assets by asset_id and count the number of client_ids (assets needed per asset type)
# This aggregation shows how many assets of each type need to be ordered
assets_to_order = assets_to_order_now.groupby('asset_id')['client_id'].count().reset_index()

# Rename the client_id column to asset_needed to better represent the count
assets_to_order.rename(columns={'client_id':'asset_needed_within_next_leadtime'}, inplace=True)
del assets_to_order_now

assets_to_order.head()

,asset_id,asset_needed_within_next_leadtime
0,AID_002,2
1,AID_003,1
2,AID_005,1
3,AID_006,2
4,AID_008,3


In [9]:
# merging the latest inventory status to the Ordering requirements
procurement_df = assets_to_order.merge(latest_inventory, how='left', on='asset_id')

del assets_to_order, latest_inventory

procurement_df.head()

,asset_id,asset_needed_within_next_leadtime,device_type,manufacturer,model_number,warehouse_location,current_stock_quantity,reorder_threshold_quantity,unit_cost,safety_stock_quantity,lead_time_days,file_year
0,AID_002,2,Server,Dell,M002,WH1,64,40,4683.44,12,10,2025
1,AID_003,1,Server,Dell,M003,WH2,66,40,788.73,12,6,2025
2,AID_005,1,Server,Cisco,M001,WH2,92,40,4531.19,28,9,2025
3,AID_006,2,Server,Cisco,M002,WH1,90,40,3738.39,20,9,2025
4,AID_008,3,Server,Cisco,M004,WH2,72,40,3434.12,21,13,2025


In [10]:
# # simulation od low stock (to be removed)
# procurement_df['current_stock_quantity'] = (procurement_df['current_stock_quantity']/40).astype(int)
# procurement_df['reorder_threshold_quantity'] = (procurement_df['reorder_threshold_quantity']/20).astype(int)
# procurement_df['safety_stock_quantity'] = (procurement_df['safety_stock_quantity']/10).astype(int)


In [11]:
# INVENTORY PROCUREMENT ALGORITHM - DETAILED LOGIC EXPLANATION
# ==============================================================
# This cell implements the core inventory management logic that determines
# which assets need to be reordered and calculates the optimal order quantities
# based on survival model predictions and current inventory levels.

# STEP 1: Calculate Inventory Deficit
# ===================================
# Logic: Current Stock - Required Buffer - Predicted Demand
#
# Required Buffer = Safety Stock
# - Reorder Threshold: Minimum level to trigger replenishment
# - Safety Stock: Extra buffer for demand variability/lead time uncertainty
# - We take the maximum to ensure we maintain the higher of the two requirements
#
# Predicted Demand = Number of assets predicted to fail within next lead time
# This comes from our survival model predictions
#
# Formula: inventory_deficit = current_stock - safety_stock - predicted_demand
#
# Interpretation:
# - Positive deficit: We have excess inventory (no order needed)
# - Negative deficit: We have insufficient inventory (order needed)
# - Zero deficit: Exactly enough inventory (no order needed)

procurement_df['inventory_deficit'] = (
      procurement_df['current_stock_quantity']
    - procurement_df['safety_stock_quantity']
    - procurement_df['asset_needed_within_next_leadtime']
)

# STEP 2: Determine Order Flag
# ============================
# Logic: Flag assets that need reordering based on deficit analysis
#
# Condition: inventory_deficit <= 0
# - If deficit is negative: Current stock insufficient to meet predicted demand
# - If deficit is zero: Current stock exactly meets requirements (no safety margin)
# - Only positive deficits indicate sufficient inventory with safety margin
#
# This creates a boolean flag for easy filtering of assets requiring attention

procurement_df['to_order_flag'] = procurement_df['inventory_deficit'] <= 0

# STEP 3: Calculate Order Quantity
# ================================
# Logic: Calculate how much to order for assets that need replenishment
#
# For assets needing reorder (to_order_flag == True):
# Required Quantity = Required Buffer + Predicted Demand - Current Stock
#
# Where Required Buffer = max(Reorder Threshold, Safety Stock)
#
# This ensures that after ordering:
# 1. We meet the predicted demand for next lead time
# 2. We maintain adequate buffer stock (reorder threshold or safety stock)
# 3. We don't over-order (only what's needed to reach target levels)
#
# For assets not needing reorder (to_order_flag == False):
# Order Quantity = 0 (no order needed)
#
# The np.where() function implements this conditional logic efficiently

procurement_df['to_order_quantity'] = np.where(
    procurement_df['to_order_flag'] == True,
    (
          procurement_df['safety_stock_quantity']
        + procurement_df['asset_needed_within_next_leadtime']
        - procurement_df['current_stock_quantity']
    ),  # Calculate required quantity for items needing reorder
    0   # No order needed for items with sufficient inventory
)

# STEP 4: Calculate Post-Lead Time Inventory Level
# ================================================
# Logic: Project inventory levels after lead time and ordering
#
# Formula: after_leadtime_inventory_level =
#           current_stock - predicted_demand + order_quantity
#
# This calculation shows:
# 1. Current inventory level
# 2. Minus assets that will fail during lead time (predicted demand)
# 3. Plus new assets that will arrive from current orders
#
# Business Value:
# - Validates that ordering strategy maintains adequate inventory
# - Helps identify potential stockouts or overstock situations
# - Supports cash flow planning by showing future inventory positions

procurement_df['after_leadtime_inventory_level'] = (
    procurement_df['current_stock_quantity']
    - procurement_df['asset_needed_within_next_leadtime']
    + procurement_df['to_order_quantity']
)

# ALGORITHM SUMMARY:
# ==================
# This implementation creates a data-driven procurement system that:
# 1. Uses survival model predictions to forecast asset failures
# 2. Maintains appropriate safety buffers for risk mitigation
# 3. Optimizes order quantities to minimize carrying costs while preventing stockouts
# 4. Provides visibility into future inventory positions for planning
#
# Key Benefits:
# - Proactive: Orders before failures occur based on predictions
# - Efficient: Orders optimal quantities, not excessive amounts
# - Risk-aware: Maintains safety buffers for uncertainty
# - Cost-effective: Minimizes both stockout costs and carrying costs

In [12]:
procurement_df[[  'asset_id', 'asset_needed_within_next_leadtime', 'current_stock_quantity'
                , 'safety_stock_quantity', 'inventory_deficit'
                , 'to_order_flag', 'to_order_quantity', 'after_leadtime_inventory_level'
            ]]

,asset_id,asset_needed_within_next_leadtime,current_stock_quantity,safety_stock_quantity,inventory_deficit,to_order_flag,to_order_quantity,after_leadtime_inventory_level
0,AID_002,2,64,12,50,False,0,62
1,AID_003,1,66,12,53,False,0,65
2,AID_005,1,92,28,63,False,0,91
3,AID_006,2,90,20,68,False,0,88
4,AID_008,3,72,21,48,False,0,69
5,AID_009,1,69,27,41,False,0,68
6,AID_010,3,69,30,36,False,0,66
7,AID_011,2,175,12,161,False,0,173
8,AID_012,3,105,17,85,False,0,102
9,AID_013,2,100,30,68,False,0,98


In [13]:
# procurement_df.to_csv('procurement_final_data.csv')

In [14]:
# Save the processed predictions to disk for use in next step of Inventory Procurment ordeing Algorithm.
os.makedirs('base_data', exist_ok=True)

with open('base_data/procurement_final_data.pkl', 'wb') as f:
    pickle.dump(procurement_df, f)
print("Final shape of the procurement_final_data dataset is : ", procurement_df.shape)

Final shape of the procurement_final_data dataset is :  (57, 16)


In [15]:
# =========================
# EXPORT: Procurement Plan + KPIs to JSON
# =========================

import json
from pathlib import Path

JSON_DIR = Path("model_outputs")
JSON_DIR.mkdir(exist_ok=True)

# ── 1. Derive urgency column ──
# Urgent = needs reorder AND lead time is long (>= 14 days)
# Planning = needs reorder but lead time is short
# Optimal = no reorder needed

def calc_urgency(row):
    if not row['to_order_flag']:
        return "Optimal"
    return "Urgent" if row['lead_time_days'] >= 14 else "Planning"

procurement_df['urgency'] = procurement_df.apply(calc_urgency, axis=1)

# ── 2. Select columns for JSON ──
export_cols = [
    'asset_id', 'device_type', 'model_number', 'manufacturer',
    'current_stock_quantity', 'reorder_threshold_quantity', 'safety_stock_quantity',
    'lead_time_days', 'unit_cost',
    'asset_needed_within_next_leadtime', 'inventory_deficit',
    'to_order_flag', 'to_order_quantity', 'after_leadtime_inventory_level',
    'urgency',
]
# Only keep columns that actually exist
export_cols = [c for c in export_cols if c in procurement_df.columns]

plan = procurement_df[export_cols].to_dict(orient='records')

with open(JSON_DIR / "procurement_plan.json", "w") as f:
    json.dump(plan, f, indent=2, default=str)
print(f"✅  procurement_plan.json: {len(plan)} records → {JSON_DIR}")

# ── 3. Page-level KPIs ──
kpis = {
    "total_assets": int(procurement_df['current_stock_quantity'].sum()),
    "stock_health_pct": round(
        float((procurement_df['after_leadtime_inventory_level'] > 0).mean() * 100), 1
    ),
    "procurement_cost_mtd": float(
        (procurement_df['to_order_quantity'] * procurement_df.get('unit_cost', 0)).sum()
    ),
    "critical_lows": int(procurement_df['to_order_flag'].sum()),
}

with open(JSON_DIR / "inventory_kpis.json", "w") as f:
    json.dump(kpis, f, indent=2)
print(f"✅  inventory_kpis.json → {JSON_DIR}")

print("Done.")

✅  procurement_plan.json: 57 records → model_outputs
✅  inventory_kpis.json → model_outputs
Done.
